# 4. 선형회귀 직접 구현 — NumPy만으로

> **제4장** · **이론편 대응: 5장 (미적분과 최적화), 8.2절 (회귀), 11.6절 (에폭·배치)**
> **예상 소요**: 50분
> **필요 사양**: CPU만으로 충분

---

## 이 장에서 하는 일

이론편 5장에서 배운 경사하강법으로 **실제 데이터에 직선을 맞춘다.** 라이브러리 없이 NumPy만 쓴다.

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 2 | 손실함수 정의 (MSE) | 8.2절 |
| 3 | 그래디언트 유도와 구현 | 5.2절 |
| 4 | 학습 루프 — 첫 시도가 실패한다 | 5.4절 |
| 5 | **표준화로 문제 해결** | — |
| 6 | 정답과 대조 | — |
| 7 | 배치 방식 비교 | 11.6절 |

**4절에서 일부러 실패하는 과정을 넣었다.** 왜 표준화가 필요한지는 실패를 겪어 봐야 이해되기 때문이다.
실무에서도 이 문제로 막히는 경우가 많다.

---

## 1. 문제 설정 — 아파트 면적으로 가격 예측

이론편 8.2절에서 회귀의 예로 들었던 문제를 실제 데이터로 만든다.

$$\hat{y} = wx + b$$

- $x$: 아파트 면적(㎡)
- $y$: 매매가(억 원)
- $w$, $b$: 우리가 찾아야 할 값

데이터는 인위적으로 만든다. **참값을 알고 있어야 결과가 맞는지 판단할 수 있기 때문**이다.
실제 데이터는 8장부터 다룬다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

# 한글 폰트 (03장 참조)
_cands = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
          "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_avail = {f.name for f in fm.fontManager.ttflist}
for _n in _cands.get(platform.system(), []):
    if _n in _avail:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

# --- 데이터 생성 ---
rng = np.random.RandomState(42)
n = 100

X = rng.uniform(20, 120, n)          # 면적 20~120㎡
TRUE_W, TRUE_B = 0.45, 10.0          # 우리가 찾아야 할 정답
noise = rng.randn(n) * 6             # 현실의 잡음
y = TRUE_W * X + TRUE_B + noise

print("=" * 50)
print("데이터")
print("=" * 50)
print(f"개수      : {n}")
print(f"면적 범위  : {X.min():.1f} ~ {X.max():.1f} ㎡")
print(f"가격 범위  : {y.min():.1f} ~ {y.max():.1f} 억")
print(f"참값      : w = {TRUE_W}, b = {TRUE_B}")
print()
print("앞 5개 데이터")
for i in range(5):
    print(f"  면적 {X[i]:6.1f}㎡ → {y[i]:6.1f}억")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(X, y, alpha=0.6, s=30, color="#1E40AF")
ax.set_xlabel("면적 (㎡)")
ax.set_ylabel("매매가 (억 원)")
ax.set_title("우리가 직선을 맞춰야 할 데이터")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---

## 2. 손실함수 — 이론편 8.2절

"직선이 데이터를 얼마나 잘 설명하는가"를 숫자 하나로 나타내야 한다.
이론편 8.2절에서 다룬 평균제곱오차(MSE)를 쓴다.

$$L(w, b) = \frac{1}{n}\sum_{i=1}^{n}\left(\hat{y}_i - y_i\right)^2 = \frac{1}{n}\sum_{i=1}^{n}\left(wx_i + b - y_i\right)^2$$

이론편에서 짚었듯 제곱을 하는 이유는 두 가지다 — **부호 상쇄를 막고**, **큰 오차에 무거운 벌점**을 주기 위해서다.

In [ ]:
import numpy as np

def predict(X, w, b):
    """직선 모델의 예측값"""
    return w * X + b

def mse_loss(X, y, w, b):
    """평균제곱오차 (이론편 8.2절)"""
    pred = predict(X, w, b)
    return np.mean((pred - y) ** 2)

# 이론편 8.2절의 아파트 3채 예제로 함수 검증
X_test = np.array([1.0, 1.0, 1.0])          # 편의상 x=1
y_test = np.array([3.0, 5.0, 4.0])          # 실제 가격
pred_test = np.array([2.5, 5.5, 4.2])       # 예측

# w=0, b를 쓰지 않고 직접 계산해 확인
manual = np.mean((pred_test - y_test) ** 2)
print("=" * 50)
print("손실함수 검증 (이론편 8.2절)")
print("=" * 50)
print(f"실제   : {y_test}")
print(f"예측   : {pred_test}")
print(f"오차   : {(pred_test - y_test).round(2)}")
print(f"제곱   : {((pred_test - y_test)**2).round(3)}")
print(f"MSE    : {manual:.4f}")
print(f"이론편 값 : 0.18")
assert abs(manual - 0.18) < 1e-6
print("[OK] 이론편 8.2절 손계산과 일치")

# 실제 데이터에서 몇 가지 (w, b) 조합의 손실
print()
print("우리 데이터에서 여러 (w, b)의 손실")
print("-" * 50)
for w, b in [(0.0, 0.0), (0.2, 5.0), (0.45, 10.0), (0.8, 20.0)]:
    L = mse_loss(X, y, w, b)
    mark = "  ← 참값" if (w, b) == (0.45, 10.0) else ""
    print(f"  w={w:<6} b={b:<6} → 손실 {L:10.2f}{mark}")

---

## 3. 그래디언트 — 이론편 5.2절

손실을 줄이려면 어느 방향으로 $w$와 $b$를 움직여야 할까. 이론편 5.2절에서 다룬 편미분으로 구한다.

**$w$에 대한 편미분**

$$\frac{\partial L}{\partial w} = \frac{1}{n}\sum_i 2\left(wx_i + b - y_i\right) \cdot x_i = \frac{2}{n}\sum_i (\hat{y}_i - y_i)\,x_i$$

바깥 제곱을 미분해 2가 나오고, 연쇄법칙으로 안쪽을 $w$로 미분한 $x_i$가 곱해진다.

**$b$에 대한 편미분**

$$\frac{\partial L}{\partial b} = \frac{2}{n}\sum_i (\hat{y}_i - y_i)$$

안쪽을 $b$로 미분하면 1이므로 $x_i$가 없다.

두 식 모두 **(예측 − 실제)** 를 공통으로 갖는다. 이것이 오차이고, 경사하강법은 결국
"오차가 큰 쪽으로 더 크게 움직인다"는 규칙이다.

In [ ]:
import numpy as np

def gradients(X, y, w, b):
    """손실함수의 편미분 (이론편 5.2절)"""
    n = len(X)
    pred = predict(X, w, b)
    error = pred - y                       # 두 식의 공통 부분
    dw = 2 * np.mean(error * X)
    db = 2 * np.mean(error)
    return dw, db


# --- 수치 미분으로 검증 ---
# 이론편 5.1절의 정의: (L(w+h) - L(w)) / h
def numeric_gradients(X, y, w, b, h=1e-5):
    dw = (mse_loss(X, y, w + h, b) - mse_loss(X, y, w - h, b)) / (2 * h)
    db = (mse_loss(X, y, w, b + h) - mse_loss(X, y, w, b - h)) / (2 * h)
    return dw, db


w_test, b_test = 0.3, 5.0
dw_a, db_a = gradients(X, y, w_test, b_test)
dw_n, db_n = numeric_gradients(X, y, w_test, b_test)

print("=" * 55)
print("그래디언트 검증 (수식 vs 수치 미분)")
print("=" * 55)
print(f"검증 지점: w={w_test}, b={b_test}")
print()
print(f"{'':10}{'수식 유도':<18}{'수치 미분':<18}{'차이'}")
print("-" * 55)
print(f"{'dL/dw':10}{dw_a:<18.6f}{dw_n:<18.6f}{abs(dw_a-dw_n):.2e}")
print(f"{'dL/db':10}{db_a:<18.6f}{db_n:<18.6f}{abs(db_a-db_n):.2e}")
print("-" * 55)

assert abs(dw_a - dw_n) < 1e-3
assert abs(db_a - db_n) < 1e-3
print("[OK] 손으로 유도한 식이 맞다")
print()
print("두 값이 모두 양수 → w와 b를 줄여야 손실이 준다는 뜻")

### 수치 미분으로 검증하는 습관

방금 한 것이 **그래디언트 체크**다. 손으로 유도한 미분식이 맞는지 이론편 5.1절의 정의로 확인하는 것이다.

$$\frac{\partial L}{\partial w} \approx \frac{L(w+h) - L(w-h)}{2h}$$

수치 미분은 느려서 실제 학습에는 쓸 수 없지만, **구현이 맞는지 확인하는 데는 유용하다.**
12장에서 역전파를 직접 구현할 때 이 방법을 다시 쓴다.

---

## 4. 첫 시도 — 그리고 실패

이제 이론편 5.4절의 갱신 규칙을 그대로 적용한다.

$$w \leftarrow w - \eta\frac{\partial L}{\partial w}, \qquad b \leftarrow b - \eta\frac{\partial L}{\partial b}$$

학습률 $\eta$를 0.001로 잡고 50번 돌려 보자. **결과가 어떻게 되는지 보라.**

In [ ]:
import numpy as np

def train(X, y, lr, n_steps, w0=0.0, b0=0.0, verbose_every=None):
    """경사하강법으로 w, b를 학습한다 (이론편 5.4절)"""
    w, b = w0, b0
    history = {"w": [], "b": [], "loss": []}

    for step in range(n_steps):
        loss = mse_loss(X, y, w, b)
        history["w"].append(w)
        history["b"].append(b)
        history["loss"].append(loss)

        if verbose_every and step % verbose_every == 0:
            print(f"  스텝 {step:5d}: w={w:12.4f}  b={b:12.4f}  손실={loss:15.4f}")

        if not np.isfinite(loss):
            print(f"  스텝 {step}: 손실이 발산했습니다.")
            break

        dw, db = gradients(X, y, w, b)
        w -= lr * dw
        b -= lr * db

    return w, b, history


print("=" * 70)
print("시도 1: 학습률 0.001, 50스텝")
print("=" * 70)
w1, b1, h1 = train(X, y, lr=0.001, n_steps=50, verbose_every=10)
print()
print(f"최종: w={w1:.4e}, b={b1:.4e}")
print(f"참값: w={TRUE_W}, b={TRUE_B}")

### 무슨 일이 일어났나

값이 터무니없이 커졌다. **발산**한 것이다. 이론편 5.4절에서 "학습률이 너무 크면 최솟값을 지나쳐
진동하거나 발산한다"고 했던 바로 그 현상이다.

그런데 0.001이면 작은 값 아닌가? **문제는 학습률 자체가 아니라 데이터의 크기에 있다.**

그래디언트 식을 다시 보자.

$$\frac{\partial L}{\partial w} = \frac{2}{n}\sum_i (\hat{y}_i - y_i)\,x_i$$

여기에 $x_i$가 곱해져 있다. 우리 데이터에서 $x$는 20~120 범위이므로, **그래디언트가 $x$의 크기만큼 증폭된다.**
반면 $b$의 그래디언트에는 $x$가 없다.

즉 $w$ 방향과 $b$ 방향의 그래디언트 크기가 크게 다르다. 이는 이론편 11.1절에서 본
**한쪽으로 길쭉한 골짜기** 상황과 같다.

In [ ]:
import numpy as np

print("=" * 55)
print("두 방향의 그래디언트 크기 비교")
print("=" * 55)
dw, db = gradients(X, y, 0.0, 0.0)
print(f"시작점 (w=0, b=0)에서")
print(f"  dL/dw = {dw:12.2f}")
print(f"  dL/db = {db:12.2f}")
print(f"  비율  = {abs(dw/db):12.1f}배")
print()
print("→ w 방향 그래디언트가 훨씬 크다.")
print("  w에 맞춰 학습률을 줄이면 b가 너무 느리고,")
print("  b에 맞추면 w가 발산한다.")
print()

print("=" * 55)
print("시도 2: 학습률을 크게 줄여 본다 (0.0001)")
print("=" * 55)
w2, b2, h2 = train(X, y, lr=0.0001, n_steps=2000)
print(f"2000스텝 후: w={w2:.4f}, b={b2:.4f}, 손실={h2['loss'][-1]:.2f}")
print(f"참값       : w={TRUE_W},   b={TRUE_B}")
print()
print("w는 얼추 맞았지만 b가 한참 모자라다.")
print("2000스텝을 돌렸는데도 이 정도다.")

---

## 5. 해결 — 입력을 표준화한다

문제의 원인이 "$x$의 크기가 커서 그래디언트가 증폭된다"는 것이었으니,
**$x$의 크기를 조정하면 된다.**

가장 흔히 쓰는 방법이 표준화(standardization)다.

$$x_{scaled} = \frac{x - \mu}{\sigma}$$

평균을 빼서 중심을 0으로 옮기고, 표준편차로 나눠 퍼짐 정도를 1로 맞춘다.
그러면 $x$가 대략 −2 ~ 2 범위에 들어온다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 표준화
X_mean, X_std = X.mean(), X.std()
X_scaled = (X - X_mean) / X_std

print("=" * 50)
print("표준화 전후 비교")
print("=" * 50)
print(f"{'':12}{'평균':<12}{'표준편차':<12}{'범위'}")
print("-" * 50)
print(f"{'원본':12}{X.mean():<12.2f}{X.std():<12.2f}{X.min():.1f} ~ {X.max():.1f}")
print(f"{'표준화 후':12}{X_scaled.mean():<12.2f}{X_scaled.std():<12.2f}{X_scaled.min():.2f} ~ {X_scaled.max():.2f}")
print()

# 그래디언트 크기가 어떻게 변했는지
dw_s, db_s = gradients(X_scaled, y, 0.0, 0.0)
print("시작점에서의 그래디언트")
print(f"  원본     : dL/dw = {gradients(X, y, 0, 0)[0]:10.2f},  dL/db = {gradients(X, y, 0, 0)[1]:8.2f}")
print(f"  표준화 후 : dL/dw = {dw_s:10.2f},  dL/db = {db_s:8.2f}")
print(f"  비율      : {abs(gradients(X,y,0,0)[0]/gradients(X,y,0,0)[1]):.1f}배 → {abs(dw_s/db_s):.1f}배")
print()
print("→ 두 방향의 크기가 비슷해졌다. 이제 하나의 학습률로 둘 다 다룰 수 있다.")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(X, bins=20, color="#1E40AF", alpha=0.7)
axes[0].set_title("원본 x 분포")
axes[0].set_xlabel("면적 (㎡)")
axes[1].hist(X_scaled, bins=20, color="#0D9488", alpha=0.7)
axes[1].set_title("표준화 후 x 분포")
axes[1].set_xlabel("표준화된 값")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

print("=" * 60)
print("시도 3: 표준화한 데이터로 학습 (학습률 0.1)")
print("=" * 60)
w3, b3, h3 = train(X_scaled, y, lr=0.1, n_steps=200, verbose_every=40)
print()
print(f"최종 (표준화 공간): w={w3:.4f}, b={b3:.4f}")
print(f"최종 손실         : {h3['loss'][-1]:.4f}")
print()
print("200스텝만에 수렴했다. 앞에서 2000스텝으로도 안 됐던 것과 비교된다.")

### 표준화 공간의 값을 원래 스케일로 되돌리기

학습은 표준화된 공간에서 했으므로, 결과를 원래 단위로 환산해야 한다.

표준화된 모델은 다음과 같다.

$$\hat{y} = w_s \cdot \frac{x - \mu}{\sigma} + b_s$$

이를 $x$에 대해 정리하면,

$$\hat{y} = \frac{w_s}{\sigma}\,x + \left(b_s - \frac{w_s \mu}{\sigma}\right)$$

따라서 원래 스케일의 계수는 다음과 같다.

$$w = \frac{w_s}{\sigma}, \qquad b = b_s - \frac{w_s\,\mu}{\sigma}$$

In [ ]:
import numpy as np

# 원래 스케일로 환산
w_final = w3 / X_std
b_final = b3 - w3 * X_mean / X_std

print("=" * 55)
print("최종 결과")
print("=" * 55)
print(f"{'':16}{'w':<14}{'b'}")
print("-" * 55)
print(f"{'학습 결과':16}{w_final:<14.4f}{b_final:.4f}")
print(f"{'참값':16}{TRUE_W:<14.4f}{TRUE_B:.4f}")
print("-" * 55)
print()
print(f"최종 손실 (원 스케일): {mse_loss(X, y, w_final, b_final):.4f}")
print()
print("참값과 조금 다른 것은 데이터에 잡음을 섞었기 때문이다.")
print("잡음까지 완벽히 맞히는 것은 오히려 과대적합이다 (이론편 8.5절).")

---

## 6. 정답과 대조 — 정규방정식

선형회귀는 **정확한 해를 수식으로 구할 수 있는** 드문 경우다. 정규방정식이라 부르며,
경사하강법으로 찾은 답이 얼마나 정확한지 확인하는 기준이 된다.

$$\mathbf{w}^* = (X^\top X)^{-1}X^\top \mathbf{y}$$

NumPy에는 이를 안정적으로 계산하는 `np.linalg.lstsq`가 있다.

In [ ]:
import numpy as np

# 설계 행렬: [x, 1] 형태로 b도 함께 다룬다
X_design = np.stack([X, np.ones(len(X))], axis=1)
optimal, *_ = np.linalg.lstsq(X_design, y, rcond=None)
w_opt, b_opt = optimal

print("=" * 60)
print("경사하강법 vs 정규방정식(정확해)")
print("=" * 60)
print(f"{'':20}{'w':<14}{'b':<14}{'손실'}")
print("-" * 60)
print(f"{'경사하강법':20}{w_final:<14.6f}{b_final:<14.6f}{mse_loss(X,y,w_final,b_final):.6f}")
print(f"{'정규방정식':20}{w_opt:<14.6f}{b_opt:<14.6f}{mse_loss(X,y,w_opt,b_opt):.6f}")
print("-" * 60)
print(f"{'차이':20}{abs(w_final-w_opt):<14.2e}{abs(b_final-b_opt):<14.2e}")
print()

assert abs(w_final - w_opt) < 1e-3, "최적해와 차이가 큽니다"
assert abs(b_final - b_opt) < 1e-2, "최적해와 차이가 큽니다"
print("[OK] 경사하강법이 정확해에 도달했다")
print()
print("그렇다면 왜 굳이 경사하강법을 쓸까?")
print("  - 정규방정식은 역행렬 계산이 필요해 특성이 많으면 매우 느려진다")
print("  - 신경망처럼 비선형인 경우에는 아예 쓸 수 없다")
print("  - 경사하강법은 어떤 모델에도 적용된다 (이론편 5.4절)")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- (1) 학습 결과 직선 ---
ax = axes[0]
ax.scatter(X, y, alpha=0.5, s=25, color="#94A3B8", label="데이터")
x_line = np.linspace(X.min(), X.max(), 100)
ax.plot(x_line, w_final * x_line + b_final, color="#EA580C",
        linewidth=2.5, label=f"학습 결과")
ax.plot(x_line, TRUE_W * x_line + TRUE_B, color="#0D9488",
        linewidth=2, linestyle="--", label="참값")
ax.set_xlabel("면적 (㎡)")
ax.set_ylabel("매매가 (억)")
ax.set_title("맞춰진 직선")
ax.legend()
ax.grid(alpha=0.3)

# --- (2) 손실 곡선 ---
ax = axes[1]
ax.plot(h3["loss"], color="#1E40AF", linewidth=2)
ax.set_xlabel("스텝")
ax.set_ylabel("손실 (MSE)")
ax.set_title("학습 곡선")
ax.grid(alpha=0.3)

# --- (3) 손실 지형과 이동 경로 ---
ax = axes[2]
w_grid = np.linspace(-5, 30, 60)
b_grid = np.linspace(0, 80, 60)
WW, BB = np.meshgrid(w_grid, b_grid)
ZZ = np.array([[mse_loss(X_scaled, y, w, b) for w in w_grid] for b in b_grid])

cs = ax.contour(WW, BB, ZZ, levels=25, colors="#CBD5E1", linewidths=0.8)
ax.plot(h3["w"], h3["b"], color="#EA580C", linewidth=2, marker="o",
        markersize=2.5, label="이동 경로")
ax.scatter([h3["w"][0]], [h3["b"][0]], color="#1E40AF", s=80, zorder=5, label="시작")
ax.scatter([w3], [b3], color="#0D9488", s=120, marker="*", zorder=5, label="도착")
ax.set_xlabel("w (표준화 공간)")
ax.set_ylabel("b")
ax.set_title("손실 지형 위의 경로")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print("오른쪽 그림이 이론편 5.5절에서 본 등고선 그림이다.")
print("경사하강법이 골짜기를 따라 최솟값으로 내려간 자취를 볼 수 있다.")

---

## 7. 배치 방식 비교 — 이론편 11.6절

지금까지는 매 스텝마다 **전체 데이터**로 그래디언트를 구했다(전체 배치).
이론편 11.6절에서 다룬 세 가지 방식을 비교해 보자.

| 방식 | 한 번에 보는 데이터 | 이 데이터(100개)에서 |
|---|---|---|
| 전체 배치 | 100개 | 1에폭당 1회 갱신 |
| 미니배치 | 16개 | 1에폭당 7회 갱신 |
| 확률적(SGD) | 1개 | 1에폭당 100회 갱신 |

In [ ]:
import numpy as np

def train_minibatch(X, y, lr, n_epochs, batch_size, seed=42):
    """미니배치 경사하강법 (이론편 11.6절)"""
    rng = np.random.RandomState(seed)
    n = len(X)
    w, b = 0.0, 0.0
    epoch_losses = []
    n_updates = 0

    for epoch in range(n_epochs):
        # 매 에폭마다 데이터를 섞는다 (이론편 11.6절의 무작위성 중 하나)
        idx = rng.permutation(n)
        X_shuf, y_shuf = X[idx], y[idx]

        for start in range(0, n, batch_size):
            xb = X_shuf[start:start + batch_size]
            yb = y_shuf[start:start + batch_size]
            dw, db = gradients(xb, yb, w, b)
            w -= lr * dw
            b -= lr * db
            n_updates += 1

        epoch_losses.append(mse_loss(X, y, w, b))

    return w, b, epoch_losses, n_updates


print("=" * 70)
print("배치 크기별 비교 (30에폭, 학습률 0.1)")
print("=" * 70)
print(f"{'방식':<16}{'배치':<8}{'갱신 횟수':<12}{'최종 손실':<14}{'w':<10}{'b'}")
print("-" * 70)

results = {}
for name, bs in [("확률적 (SGD)", 1), ("미니배치", 16), ("전체 배치", len(X))]:
    w, b, losses, updates = train_minibatch(X_scaled, y, lr=0.1,
                                            n_epochs=30, batch_size=bs)
    results[name] = losses
    print(f"{name:<16}{bs:<8}{updates:<12}{losses[-1]:<14.4f}{w:<10.4f}{b:.4f}")

print("-" * 70)
print()
print("배치가 작을수록 갱신 횟수가 많아 같은 에폭에서 더 많이 학습한다.")
print("대신 방향이 흔들려 손실 곡선이 울퉁불퉁해진다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))

colors = {"확률적 (SGD)": "#EA580C", "미니배치": "#0D9488", "전체 배치": "#1E40AF"}
for name, losses in results.items():
    ax.plot(losses, label=name, linewidth=2, color=colors[name])

ax.set_xlabel("에폭")
ax.set_ylabel("손실 (MSE)")
ax.set_title("배치 크기에 따른 학습 곡선 (이론편 11.6절)")
ax.set_yscale("log")
ax.legend()
ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

print("세로축이 로그 눈금이라는 점에 주의한다.")
print("전체 배치는 매끄럽지만 느리고, SGD는 빠르지만 흔들린다.")
print("미니배치가 둘의 절충안이며 실무에서 가장 널리 쓰인다.")

---

## 8. 정리

### 확인한 이론편 값

| 이론편 절 | 내용 | 결과 |
|---|---|---|
| 8.2 | MSE = 0.18 (아파트 3채) | ✓ |
| 5.2 | 그래디언트 유도식 | 수치 미분과 일치 ✓ |
| 5.4 | 경사하강 수렴 | 정규방정식과 일치 ✓ |
| 11.6 | 배치 방식 3종 | 곡선으로 확인 ✓ |

### 이 장의 핵심

**입력의 크기(스케일)를 맞추지 않으면 학습이 안 된다.**

이것이 4절에서 일부러 실패해 본 이유다. 그래디언트에 $x$가 곱해지므로,
$x$가 크면 그 방향의 그래디언트만 커져 학습률을 맞추기 어려워진다.

| 상황 | 학습률 0.001 | 학습률 0.0001 | 표준화 + 0.1 |
|---|---|---|---|
| 결과 | 발산 | 2000스텝에도 미수렴 | 200스텝에 수렴 |

이 문제는 이론편 11.3절의 Batch Normalization, 11.5절의 학습률 스케줄링이 다루는 문제와
뿌리가 같다. 신경망에서는 층마다 이런 일이 벌어지기 때문이다.

### 기억할 것

| 항목 | 요점 |
|---|---|
| 그래디언트 체크 | 수치 미분으로 유도식 검증 (12장에서 재사용) |
| 표준화 | `(x - mean) / std` — 학습 전 거의 항상 필요 |
| 환산 | 표준화 공간의 계수는 원 스케일로 되돌려야 해석 가능 |
| 배치 크기 | 작으면 자주·흔들리게, 크면 드물게·안정적으로 |

### 다음 장

**5. Claude Code 연동 — AI와 함께 코딩하기** — 지금까지 만든 코드를 AI와 함께 개선해 본다.
직접 만들어 이해한 코드가 있어야 AI가 준 코드를 판단할 수 있다.